In [1]:
#Import necessary modules
import pandas as pd
import numpy as np
import ast
from scipy.stats import ranksums,wilcoxon
#kmeans
from sklearn.cluster import KMeans
from sklearn import preprocessing
from sklearn.metrics import silhouette_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
import statistics
from collections import Counter

import math
from scipy.stats import variation
from scipy.stats import iqr
from scipy import stats 

import random
from random import seed
from random import randint
from sklearn.neighbors import LocalOutlierFactor
from pyod.models.lof import LOF
from pyod.models.ocsvm import OCSVM
from sklearn.metrics import confusion_matrix
import scipy.stats
import scipy.stats as st
from scipy.stats import t

import pandas as pd
from sklearn.ensemble import IsolationForest
from statistics import median
# from sklearn.metrics import precision_score, recall_score, f1_score

In [2]:
patients=[1135, 1450, 1464, 1497, 1504, 
          1511, 1541, 1559, 1572, 1582, 
          1586, 1603, 1607, 1609, 1615, 
          1617, 1628, 1657, 1660, 1706]

In [3]:
def readData_non_adjacent(patient):
    tm = pd.read_csv(f"{patient}_non_adjacent_twice_alldata"'.csv')
#     out = np.array(tm).tolist()  
    return tm

In [4]:
def readData_adjacent(patient):
    tm = [] 
    return tm

In [5]:
# def readData_adjacent(patient):
#     tm = pd.read_csv(f"{patient}_adjacent_twice_alldata"'.csv')
#     out = np.array(tm).tolist()  
#     return tm

In [6]:
def readData_two_conditions_IF(patient,per):
    tm = pd.read_csv(f"{patient,per}_possible_visitors_two_conditions_IF_alldata"'.csv')
#     tm = pd.read_csv(f"{patient,50}_possible_visitors_two_conditions_IF_Hu"'.csv')
#     out = np.array(tm).tolist()  
    return tm

In [7]:
def readData_entrance_firing(patient):
    tm = pd.read_csv(f"{patient}_entrance_firing_alldata"'.csv')
#     out = np.array(tm).tolist()  
    return tm

In [8]:
def readData_entropy(patient):
    tm = pd.read_csv(f"{patient}_entropy_alldata"'.csv')
#     out = np.array(tm).tolist()  
    return tm

In [9]:
def read_all_info(num,per):
    patient = num
    #ours
    IForest=readData_two_conditions_IF(patient,per)
    #Hu
#     IForest_Hu=readData_IF_Hu(patient,per)
    
    #abnormal areas
    non_adjacent=readData_non_adjacent(patient)
    adjacent=readData_adjacent(patient)
    
    entrance_firing=readData_entrance_firing(patient)
    entropy=readData_entropy(patient)#pe

    #entrance_firing
    new_df_entrance_firing = entrance_firing.stack().reset_index()
    new_df_entrance_firing.columns = ['hour', 'date', 'value']
    new_df_entrance_firing=new_df_entrance_firing.sort_values(['date', 'hour'])
    
    #IF
    new_df_two_consitions_IF= IForest.stack().reset_index()
    new_df_two_consitions_IF.columns = ['hour', 'date', 'value']
    new_df_two_consitions_IF=new_df_two_consitions_IF.sort_values(['date', 'hour'])
    
    #new_df_non_adjacent
    new_df_non_adjacent= non_adjacent.stack().reset_index()
    new_df_non_adjacent.columns = ['hour', 'date', 'value']
    new_df_non_adjacent=new_df_non_adjacent.sort_values(['date', 'hour'])
    
# #      #new_df_adjacent
#     new_df_adjacent= adjacent.stack().reset_index()
#     new_df_adjacent.columns = ['hour', 'date', 'value']
#     new_df_adjacent=new_df_adjacent.sort_values(['date', 'hour'])
    new_df_adjacent = pd.DataFrame(columns=['hour', 'date', 'value'])

    #entropy
    new_df_entropy= entropy.stack().reset_index()
    new_df_entropy.columns = ['hour', 'date', 'value']
    new_df_entropy=new_df_entropy.sort_values(['date', 'hour'])
#     print(new_df_entrance_firing['value'].count(),new_df_two_consitions_IF['value'].count(),
#           new_df_non_adjacent['value'].count(),new_df_entropy['value'].count())
    return new_df_entrance_firing,new_df_two_consitions_IF,new_df_non_adjacent,new_df_adjacent,new_df_entropy
    

In [10]:
def outcomes(num,m,per):
#     print(num,per)
    new_df_entrance_firing,new_df_two_consitions_IF,new_df_non_adjacent,new_df_adjacent,new_df_entropy=read_all_info(num,per)
    new_df_two_consitions_IF= new_df_two_consitions_IF.loc[new_df_two_consitions_IF['value'] >= m]

    #my method
    
    ##<1.1> events in new_df_non_adjacent(abnormal events)+firings at door area
    Ground_truth_non_adjacent= new_df_non_adjacent.loc[new_df_non_adjacent['value'] >0]
    entrance_firing= new_df_entrance_firing.loc[new_df_entrance_firing['value'] >0]

    #merge with those have entrance firings
    merge_ground =  pd.merge(Ground_truth_non_adjacent, entrance_firing, on=['date', 'hour'],how='inner')

    merge_ground['value'] = 1
    merge_ground= merge_ground[['date', 'hour', 'value']]
    Ground_truth_non_adjacent=merge_ground #abnormal events+firings at door area
    
#     ##<1.2> events in new_df_adjacent(abnormal events)
    Ground_truth_adjacent= new_df_adjacent.loc[new_df_adjacent['value'] >0]
    #merge with those have entrance firings
    merge_ground_adjacent =  pd.merge(Ground_truth_adjacent, entrance_firing, on=['date', 'hour'],how='inner')
    merge_ground_adjacent['value'] = 1
    merge_ground_adjacent = merge_ground_adjacent[['date', 'hour', 'value']]
    Ground_truth_adjacent=merge_ground_adjacent #abnormal events
  

    Ground_truth_all = pd.merge(Ground_truth_non_adjacent, Ground_truth_adjacent, on=['date', 'hour'],how='outer')
#     Ground_truth =Ground_truth_non_adjacent
    
    
    #<2> My method (IF)
    #  merge  entrance_firing 和 new_df_two_consitions_IF 中存在的数据点
    my_visitors = pd.merge(entrance_firing, new_df_two_consitions_IF, on=['date', 'hour'],how='inner')
    # add 'value' column with value 1
    my_visitors['value'] = 1
    my_visitor=my_visitors[['date', 'hour', 'value']]
    # combined with new_df_entrance_firing

    # inner merge 保留同时在 Ground_truth 和 my_visitors 中存在的数据点
    merged_truth = pd.merge(Ground_truth_all, my_visitors, on=['date', 'hour'], how='outer')
    merged_truth['value'] = 1
    merged_truth=merged_truth[['date', 'hour', 'value']]
    merged_truths=merged_truth[['date', 'hour', 'value']].copy()
    
    
    
    #<3> entropy method
    #entropy method 保留同时在 entrance_firing 和 new_df_entropy 中存在的数据点
    merged_entropy = pd.merge(entrance_firing,new_df_entropy, on=['date', 'hour'], how='inner')
    merged_entropy['value'] = 1
    merged_entropy=merged_entropy[['date', 'hour', 'value']]
    merged_entropys=merged_entropy[['date', 'hour', 'value']].copy()
    
    #-----------------------------------------------------------------------------
    ## 找出存在于 Ground_truth_non_adjacent 中，但不存在于 merged_entropy 中的数据点 
    #cases in Ground_truth but not detect by baseline 
    
    # inner merge 保留同时在 non_adjacent 和 merged_entropy 中存在的数据点
    merged = pd.merge(Ground_truth_non_adjacent, merged_entropys, on=['date', 'hour'], how='inner')
    
    
    # inner merge 保留同时在 non_adjacent 和 merged_entropy 中存在的数据点
    merged_adjacent = pd.merge(Ground_truth_adjacent, merged_entropys, on=['date', 'hour'], how='inner')
    
    
    # inner merge 保留同时在 Ground_truth_all(all) 和 merged_entropy 中存在的数据点
    merged_all = pd.merge(Ground_truth_all, merged_entropys, on=['date', 'hour'], how='inner')
 
    
    #-----------------------------------------------------------------------------
    # 找出存在于 Ground_truth(abnormal events) 中，但不存在于 merged(entropy) 中的数据点
    Difference_non_adjacent = Ground_truth_non_adjacent[~Ground_truth_non_adjacent[['date', 'hour']].apply(tuple, axis=1).isin(merged[['date', 'hour']].apply(tuple, axis=1))]
    
    Difference_adjacent = Ground_truth_adjacent[~Ground_truth_adjacent[['date', 'hour']].apply(tuple, axis=1).isin(merged_adjacent[['date', 'hour']].apply(tuple, axis=1))]

    Difference_all = Ground_truth_all[~Ground_truth_all[['date', 'hour']].apply(tuple, axis=1).isin(merged_all[['date', 'hour']].apply(tuple, axis=1))]
    
    return my_visitor, merged_truth,merged_entropy,Ground_truth_all,Ground_truth_adjacent,Ground_truth_non_adjacent,Difference_all,Difference_adjacent,Difference_non_adjacent

In [11]:
def performance(num,m,per):
#     print(num,per)
    my_visitor,merged_truth,merged_entropy,Ground_truth_all,Ground_truth_adjacent,Ground_truth_non_adjacent,Difference_all,Difference_adjacent,Difference_non_adjacent=outcomes(num,m,per)
    new_df_entrance_firing,new_df_two_consitions_IF,new_df_non_adjacent,new_df_adjacent,new_df_entropy=read_all_info(num,per)
    
    # Select data that both have value in merged_entropy &  merged_truth
    merged_df = pd.merge(merged_entropy,merged_truth, on=['date', 'hour'], how='inner')
    tp = merged_df[(merged_df['value_x'] != 0) & (merged_df['value_y'] != 0)]
    tp = tp[['date', 'hour', 'value_x', 'value_y']]
    
    # Select data in merged_entropy but not in merged_truth
    merged_df = merged_entropy.merge(merged_truth, on=['hour', 'date'], how='left')
    fn = merged_df[(merged_df['value_x'] != 0) & merged_df['value_y'].isna()]
    fn = fn[['hour', 'date', 'value_y']]
    
    # Select data in my_visitors  but not in merged_truth
    merged_df = merged_entropy.merge(merged_truth, on=['hour', 'date'], how='right')
    fp = merged_df[(merged_df['value_x'].isna()) & merged_df['value_y']!= 0]
    fp = fp[['hour', 'date', 'value_x']]
    
    TP=tp.count()['date']
    FN=fn.count()['date']
    FP=fp.count()['date']
    TN=(new_df_entrance_firing.count()['date']-TP-FN-FP)
    return TP,FN,FP,TN

In [12]:
def compare_each(num,m,per):
    new_df_entrance_firing,new_df_two_consitions_IF,new_df_non_adjacent,new_df_adjacent,new_df_entropy=read_all_info(num,per)
    IF_visitor,merged_truth,merged_entropy,Ground_truth_all,Ground_truth_adjacent,Ground_truth_non_adjacent,Difference_all,Difference_adjacent,Difference_non_adjacent=outcomes(num,m,per)
    TP,FN,FP,TN=performance(num,m,per)
    # Mymethod
    IF_visitors=IF_visitor['date'].count()
    my_visitor=merged_truth['date'].count()
    my_non_visitor=new_df_entrance_firing.count()['date']-my_visitor

    #baseline
    entropy_visitor=merged_entropy['value'].count()
    entropy_non_visitor=new_df_entrance_firing.count()['date']-entropy_visitor
        
     #-----------------------------------------------------------------------------
    #comparision
    # Select data that both have value in merged_entropy &  merged_truth
    merged_df = pd.merge(merged_entropy,IF_visitor, on=['date', 'hour'], how='inner')
    tp0 = merged_df[(merged_df['value_x'] != 0) & (merged_df['value_y'] != 0)]
    tp0 = tp0[['date', 'hour', 'value_x', 'value_y']]
    
    merged_df = pd.merge(merged_entropy,Ground_truth_non_adjacent, on=['date', 'hour'], how='inner')
    tp1 = merged_df[(merged_df['value_x'] != 0) & (merged_df['value_y'] != 0)]
    tp1 = tp1[['date', 'hour', 'value_x', 'value_y']]
    
    merged_df = pd.merge(merged_entropy,Ground_truth_adjacent, on=['date', 'hour'], how='inner')
    tp2 = merged_df[(merged_df['value_x'] != 0) & (merged_df['value_y'] != 0)]
    tp2 = tp2[['date', 'hour', 'value_x', 'value_y']]
    
    merged_df = pd.merge(merged_entropy,merged_truth, on=['date', 'hour'], how='inner')
    tp3 = merged_df[(merged_df['value_x'] != 0) & (merged_df['value_y'] != 0)]
    tp3 = tp3[['date', 'hour', 'value_x', 'value_y']]
    
    print( #'entropy_visitor',entropy_visitor,
#          " ",
          'IF_visitors',IF_visitors,"(",IF_visitors-tp0.count()['date'],")",
         
#           'Adjacent areas',Ground_truth_adjacent['date'].count(),"(",Ground_truth_adjacent['date'].count()-tp2.count()['date'],")",
          
#           'Non-adjacent areas',Ground_truth_non_adjacent['date'].count(), "(",Ground_truth_non_adjacent['date'].count()-tp1.count()['date'],")",
#           
          'my_visitor',my_visitor, "(",my_visitor-tp3.count()['date'],")",)
    
  
    return merged_df,entropy_visitor,entropy_non_visitor,my_visitor,my_non_visitor,TP,FN,FP,TN

In [15]:
# def compare_numbers_macro(num,m,per):
#     new_df_entrance_firing,new_df_two_consitions_IF,new_df_non_adjacent,new_df_adjacent,new_df_entropy=read_all_info(num,per)
#     IF_visitor,merged_truth,merged_entropy,Ground_truth_all,Ground_truth_adjacent,Ground_truth_non_adjacent,Difference_all,Difference_adjacent,Difference_non_adjacent=outcomes(num,m,per)
#     TP,FN,FP,TN=performance(num,m,per)
#     # Mymethod
#     IF_visitors=IF_visitor['date'].count()
#     my_visitor=merged_truth['date'].count()
#     my_non_visitor=new_df_entrance_firing.count()['date']-my_visitor

#     #baseline
#     entropy_visitor=merged_entropy['value'].count()
#     entropy_non_visitor=new_df_entrance_firing.count()['date']-entropy_visitor
        
#      #-----------------------------------------------------------------------------
#     #comparision
#     # Select data that both have value in merged_entropy &  merged_truth
#     merged_df = pd.merge(merged_entropy,merged_truth, on=['date', 'hour'], how='outer')  #inner
    
#     print('entropy_visitor',entropy_visitor,#'IF_visitors',IF_visitors,
          
# #           'Adjacent areas',Ground_truth_adjacent['date'].count(),
# #           'Non-adjacent areas',Ground_truth_non_adjacent['date'].count(),
#           'my_visitor',my_visitor,'TP',TP,
    
#         "Merged Visitor events", merged_df['date'].count())
    
# #     tp = merged_df[(merged_df['value_x'] != 0) & (merged_df['value_y'] != 0)]
# #     tp = tp[['date', 'hour', 'value_x', 'value_y']].count()['date']
    
# #     ours_not_baseline = my_visitor - tp
# #     baseline_not_ours = entropy_visitor- tp

# # #     total=TP+FN+FP+TN
# #     Jaccard=round((tp)/(tp+ours_not_baseline+baseline_not_ours),3)
# #     print(my_visitor,entropy_visitor)
   
# #     print( tp,ours_not_baseline,baseline_not_ours)
# #     print(Jaccard)
#     return merged_df,entropy_visitor,entropy_non_visitor,my_visitor,my_non_visitor,TP

In [16]:
# use entropy results only
def compare_numbers_macro(num,m,per):
    new_df_entrance_firing,new_df_two_consitions_IF,new_df_non_adjacent,new_df_adjacent,new_df_entropy=read_all_info(num,per)
    IF_visitor,merged_truth,merged_entropy,Ground_truth_all,Ground_truth_adjacent,Ground_truth_non_adjacent,Difference_all,Difference_adjacent,Difference_non_adjacent=outcomes(num,m,per)
    TP,FN,FP,TN=performance(num,m,per)
    # Mymethod
    IF_visitors=IF_visitor['date'].count()
    my_visitor=merged_truth['date'].count()
    my_non_visitor=new_df_entrance_firing.count()['date']-my_visitor

    #baseline
    entropy_visitor=merged_entropy['value'].count()
    entropy_non_visitor=new_df_entrance_firing.count()['date']-entropy_visitor
        
     #-----------------------------------------------------------------------------
    #comparision
    # Select data that both have value in merged_entropy &  merged_truth
    merged_df = pd.merge(merged_entropy,merged_truth, on=['date', 'hour'], how='outer')  #inner
    
    print('entropy_visitor',entropy_visitor,#'IF_visitors',IF_visitors,
          
#           'Adjacent areas',Ground_truth_adjacent['date'].count(),
#           'Non-adjacent areas',Ground_truth_non_adjacent['date'].count(),
          'my_visitor',my_visitor,'TP',TP,
    
        "Merged Visitor events", merged_df['date'].count())
    
#     tp = merged_df[(merged_df['value_x'] != 0) & (merged_df['value_y'] != 0)]
#     tp = tp[['date', 'hour', 'value_x', 'value_y']].count()['date']
    
#     ours_not_baseline = my_visitor - tp
#     baseline_not_ours = entropy_visitor- tp

# #     total=TP+FN+FP+TN
#     Jaccard=round((tp)/(tp+ours_not_baseline+baseline_not_ours),3)
#     print(my_visitor,entropy_visitor)
   
#     print( tp,ours_not_baseline,baseline_not_ours)
#     print(Jaccard)
    return merged_df,entropy_visitor,entropy_non_visitor,my_visitor,my_non_visitor,TP

In [19]:
##best
m=1
per=50

for num in patients:
    patient = num

    merged_df,entropy_visitor,entropy_non_visitor,my_visitor,my_non_visitor,TP=compare_numbers_macro(num,m,per)

    # My method (merged_truth) to a df and Sort by 'date' and 'hour'
    df=merged_df[['date', 'hour']]
    
    df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y')


    # Sort the DataFrame by the 'date' column
    df.sort_values(by='date', inplace=True)
    
    df.to_csv(f"{patient}_Ensemblel_visitor_alldata_50th"'.csv', index=False, header=True)#_Ensemblel_visitor_alldata_75th
#     df.to_csv(f"{patient}_Entropy_visitor_alldata"'.csv', index=False, header=True)

entropy_visitor 2507 my_visitor 3679 TP 1829 Merged Visitor events 4357


<ipython-input-19-8dbfdc1b5639>:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y')
<ipython-input-19-8dbfdc1b5639>:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.sort_values(by='date', inplace=True)


entropy_visitor 3692 my_visitor 3465 TP 2427 Merged Visitor events 4730


<ipython-input-19-8dbfdc1b5639>:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y')
<ipython-input-19-8dbfdc1b5639>:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.sort_values(by='date', inplace=True)


entropy_visitor 2661 my_visitor 3231 TP 2030 Merged Visitor events 3862


...

<ipython-input-19-8dbfdc1b5639>:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y')
<ipython-input-19-8dbfdc1b5639>:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.sort_values(by='date', inplace=True)


entropy_visitor 1263 my_visitor 1522 TP 696 Merged Visitor events 2089


<ipython-input-19-8dbfdc1b5639>:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y')
<ipython-input-19-8dbfdc1b5639>:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.sort_values(by='date', inplace=True)


entropy_visitor 910 my_visitor 1286 TP 596 Merged Visitor events 1600


<ipython-input-19-8dbfdc1b5639>:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y')
<ipython-input-19-8dbfdc1b5639>:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.sort_values(by='date', inplace=True)


In [20]:
xx

NameError: name 'xx' is not defined

In [ ]:
# m=1
# per=50
# for num in [1,2,4,7,105,111,114,120,123,127,130]:#

#     tp,entropy_visitor,entropy_non_visitor,my_visitor,my_non_visitor=compare_numbers_macro(num,m,per)
# #     print(num,"visitor",entropy_visitor,entropy_non_visitor,my_visitor,my_non_visitor)
#     print(num, tp['date'].count())

   
# #     print(num,Ground_truth_all['date'].count(),Difference_all['date'].count(), round(1-(Difference_all['date'].count()/Ground_truth_all['date'].count()),2),
# #           Ground_truth_adjacent['date'].count(),Difference_adjacent['date'].count(),round(1-(Difference_adjacent['date'].count()/ Ground_truth_adjacent['date'].count()),2),
# #           Ground_truth_non_adjacent['date'].count(), Difference_non_adjacent['date'].count(),round(1-(Difference_non_adjacent['date'].count()/Ground_truth_non_adjacent['date'].count()),2))
    
# #     print(round(TPR,2),round(TNR,2))
    
# #     print(my_visitor,round(FP_V,2),round(1-round(FP_V,2)/my_visitor,2))
# #     print(num,TP_V,FN_V,FP_V,TN_V,round(MCC,2), round(G_mean,2),round(macro_avg_f1_score,2),round(TPR,2),round(TNR,2))
# #     print(num,"Total abnormal events",Ground_truth['date'].count(), "but not detected by the entropy method",Ground_truth_only_all['date'].count())
# #     print("detected by adjacent areas",Ground_truth_adjacent['date'].count(),"but not detected by the entropy method" ,Ground_truth_only_all['date'].count()-Ground_truth_only['date'].count())
# #     print("detected by non-adjacent areas", Ground_truth_non_adjacent['date'].count(),"but not detected by the entropy method", Ground_truth_only['date'].count())
# #     print("Accuracy",Accuracy)
# #     print("Macro Average Precision",macro_avg_precision)
# #     print("Macro Average Recall",macro_avg_recall)
# #     print("Macro Average F1-Score",macro_avg_f1_score,"\n")

In [ ]:
m=1
per=50
nums=[4]#1,2,4,7,105,111,114,120,123,127,130
from matplotlib.backends.backend_pdf import PdfPages
with PdfPages(f'Non-Adjacent Abnormal Events Detected by Our method with threshold equal to {per}.pdf') as pdf:

    for num in nums:
#         merged_truth,merged_entropy,Ground_truth_all,Ground_truth_adjacent,Ground_truth_non_adjacent,Difference_all,Difference_adjacent,Difference_non_adjacent=outcomes(num,m,per)
        IF_visitor,merged_truth,merged_entropy,Ground_truth_all,Ground_truth_adjacent,Ground_truth_non_adjacent,Difference_all,Difference_adjacent,Difference_non_adjacent=outcomes(num,m,per)
                #Group by hour and count the number of non-adjacent events
        if Ground_truth_non_adjacent['hour'].count()!=0:
            hourly_counts = Ground_truth_non_adjacent.groupby('hour').size()
            # Plot the histogram
    #         plt.figure(figsize=(8, 5))
            hourly_counts.plot(kind='bar', edgecolor='black')
            plt.title(f'Number of Abnormal Non-Adjacent Events in 24 Hours for user{num}')
            plt.xlabel('Hour of the Day')
            plt.ylabel('Number of Events')
            plt.xticks(rotation=0)
            plt.yticks(range(0, 50,2)) 
            plt.grid(axis='y', linestyle='--', alpha=0.6)

            pdf.savefig()
#             plt.show()
            plt.close()
        
#         if merged_truth['hour'].count()!=0:
#             hourly_counts = merged_truth.groupby('hour').size()
#             # Plot the histogram
#             plt.figure(figsize=(8, 5))
#             hourly_counts.plot(kind='bar', edgecolor='black')
#             plt.title(f'Number of All Abnormal Events Detected by Our method in 24 Hours for user{num}')
#             plt.xlabel('Hour of the Day')
#             plt.ylabel('Number of Events')
#             plt.xticks(rotation=0)
#             plt.yticks(range(0, 55,5 )) 
#             plt.grid(axis='y', linestyle='--', alpha=0.6)
#             pdf.savefig()
#             plt.close()


#         if merged_entropy['hour'].count()!=0:
#             hourly_counts = merged_entropy.groupby('hour').size()
#             # Plot the histogram
#             plt.figure(figsize=(8, 5))
#             hourly_counts.plot(kind='bar', edgecolor='black')
#             plt.title(f'Number of All Abnormal Events Detected by Baseline method in 24 Hours for user{num}')
#             plt.xlabel('Hour of the Day')
#             plt.ylabel('Number of Events')
#             plt.xticks(rotation=0)
#             plt.yticks(range(0, 35,5)) 
#             plt.grid(axis='y', linestyle='--', alpha=0.8)
#             pdf.savefig()
#             plt.close()

In [ ]:
def average_outputs(data,m):
    # Extract values when m=1 for each user
    m1_values = [user_data[m] for user_data in data.values()]

    # Transpose the data to work with percentiles as columns
    percentile_data = list(zip(*m1_values))

    # Calculate the average for each percentile
    average_percentiles = [sum(percentile) / len(percentile) for percentile in percentile_data]

    # Print the results
    print(f"Average Outputs when m={m}:")
    for i, avg_value in enumerate(average_percentiles, start=0):
        print(f"Percentile {x_values[i]}: {avg_value:.2f}")
    return percentile_data

In [ ]:
x_values = [50,75,90,95,99]

In [ ]:
MergedP_1=average_outputs(MergedP,1)

In [ ]:
MergedP_2=average_outputs(MergedP,2)

In [ ]:
MergedP_1

In [ ]:
MergedP_2

* Wilcoxon rank-sum test would be used to compare the mathematics scores in Class 1 and Class 2, 
* Wilcoxon signed-rank test can be used to compare the mathematics and English scores in Class 1 students.

In [ ]:
def signed_rank_test_user(f_score_1,f_score_2,per):
    for i in range(len(x_values)):
        if per==x_values[i]:
            # Perform the Wilcoxon signed-rank test
            statistic, p_value = scipy.stats.wilcoxon(f_score_1[i],f_score_2[i])#percentile=50

            # Print the results
        #         print(f"Wilcoxon signed-rank test statistic: {statistic}")
            print(f"P-value: {p_value}")

            # Make a decision based on the p-value
            alpha = 0.05  # Significance level
            if p_value < alpha:
                print("Reject the null hypothesis. There is a significant difference.")
            else:
                print("There is no significant difference.")


In [ ]:
signed_rank_test_user(MergedP_1,MergedP_2,75)

In [ ]:
def signed_rank_test(f_score_1):#ranksums
    import scipy.stats

    # Assuming you have two lists of F1 scores for parameter=1 and parameter=2 for each user
    f1_scores_param1 = f_score_1[1] #per=50
    f1_scores_param2 = f_score_1[2] #per=75
#     print(f1_scores_param1,f1_scores_param2)
#     # Calculate the differences
#     differences = [score1 - score2 for score1, score2 in zip(f1_scores_param1, f1_scores_param2)]

    # Perform the Wilcoxon signed-rank test
    statistic, p_value = scipy.stats.wilcoxon(f1_scores_param1,f1_scores_param2)

    # Print the results
#     print(f"Wilcoxon ranksums test statistic: {statistic}")
    print(f"P-value: {p_value}")

    # Make a decision based on the p-value
    alpha = 0.05  # Significance level
    if p_value < alpha:
        print("Reject the null hypothesis. There is a significant difference.")
    else:
        print("Fail to reject the null hypothesis. There is no significant difference.")


In [ ]:
signed_rank_test(MergedP_1)